In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Carregar o dataset a partir da pasta data/
df = pd.read_csv('../data/cybersecurity_intrusion_data.csv')

# 2. Verificar valores em falta (Missing Values) por coluna
print("--- Valores em Falta por Coluna ---")
print(df.isnull().sum())

# 3. Splitting Data: Separar Features (X) e Target (y)
X = df.drop(columns=['session_id', 'attack_detected'])
y = df['attack_detected']

# Divisão Treino / Teste (80% Treino, 20% Teste) mantendo a proporção da target (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nConjunto de Treino: {X_train.shape[0]} amostras")
print(f"Conjunto de Teste:  {X_test.shape[0]} amostras")

--- Valores em Falta por Coluna ---
session_id                0
network_packet_size       0
protocol_type             0
login_attempts            0
session_duration          0
encryption_used        1966
ip_reputation_score       0
failed_logins             0
browser_type              0
unusual_time_access       0
attack_detected           0
dtype: int64

Conjunto de Treino: 7629 amostras
Conjunto de Teste:  1908 amostras


In [2]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pickle

# 1. Separar variáveis numéricas e categóricas
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

# 2. Pipeline Numérico: Imputação com a mediana + Normalização (StandardScaler)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. Pipeline Categórico: Imputação com o valor mais frequente + One-Hot Encoding
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4. Combinar os dois pipelines num ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# 5. Aplicar o pré-processamento aos dados de treino e teste
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("--- Pré-Processamento Concluído ---")
print(f"Dimensão de X_train preparado: {X_train_prep.shape}")
print(f"Dimensão de X_test preparado:  {X_test_prep.shape}")

# 6. Guardar os conjuntos pré-processados na pasta models/ para usar no Notebook 3
with open('../models/processed_data.pkl', 'wb') as f:
    pickle.dump((X_train_prep, X_test_prep, y_train, y_test), f)

print("\nDados guardados com sucesso em '../models/processed_data.pkl'!")

--- Pré-Processamento Concluído ---
Dimensão de X_train preparado: (7629, 16)
Dimensão de X_test preparado:  (1908, 16)

Dados guardados com sucesso em '../models/processed_data.pkl'!


C:\Users\hugod\AppData\Local\Temp\ipykernel_22768\1748603742.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include=['object']).columns


In [2]:
import sys
sys.path.append('../scripts')
from data_prep import load_and_preprocess_data

# Executar a pipeline de preparação através do módulo em scripts/
X_train_prep, X_test_prep, y_train, y_test, preprocessor = load_and_preprocess_data(
    csv_path='../data/cybersecurity_intrusion_data.csv',
    save_path='../models/processed_data.pkl'
)